# Demo: Event Summarization using SmolVLM

This notebook provides a demonstration for the second task of the assignment. Given the strong performance of vision-language models across a variety of tasks, we use an efficient, lightweight model: [SmolVLM 2](https://huggingface.co/blog/smolvlm2)
The goal of this notebook is to guide you on how to: design prompts for event detection, process model outputs, and evaluate the quality of detected events.
You are expected to build upon this baseline and critically evaluate your chosen approaches.

In [26]:
!pip install num2words

Python(22364) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable


In [27]:
!pip install av


Python(22365) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable


SmolVLM2 is a lightweight vision-language model designed for efficient video understanding with relatively few parameters. The material presented here is largely based on the original documentation and is intended to provide a starting point for developing your own approach.

In [28]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import cv2
from PIL import Image
import random
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    device_map="auto",
    trust_remote_code=True,
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.85s/it]
Some parameters are on the meta device because they were offloaded to the disk.


Since SmolVLM2 is a vision-language model (VLM), you will need to design an appropriate prompt for the task. The prompt can be flexible and should be adapted based on the type of output you want (e.g., event lists, descriptions, or timestamps).

Vision-language models process visual inputs (such as frames or videos) together with text by encoding them into a shared representation space. The exact input format may vary across models.

In our case, the HuggingFace implementation handles most of the input preprocessing (e.g., frame sampling and formatting), allowing you to focus primarily on prompt design and output parsing.

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "url": "video_22.mp4"},
            {"type": "text", "text": "Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take"}
        ]
    },
]

inputs = processor.apply_chat_template(
    messages,
    num_frames = 15,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make
generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=128)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)


print(generated_texts[0])

/Users/spopa/Library/Python/3.9/lib/python/site-packages/transformers/video_processing_utils.py:879: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/Users/spopa/Library/Python/3.9/lib/python/site-packages/transformers/video_utils.py:524: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
/Users/spopa/Library/Python/3.9/lib/python/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.war

User: You are provided the following series of fifteen frames from a 0:03:55 [H:MM:SS] video.

Frame from 00:01:
Frame from 00:17:
Frame from 00:34:
Frame from 00:51:
Frame from 01:07:
Frame from 01:24:
Frame from 01:41:
Frame from 01:57:
Frame from 02:14:
Frame from 02:31:
Frame from 02:48:
Frame from 03:04:
Frame from 03:21:
Frame from 03:38:
Frame from 03:54:

Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take
Assistant: The video begins with a group of people exercising on a brick wall. One person is doing push-ups, while another is doing sit-ups. The scene then shifts to a group of people sitting on a bench, with one person standing and talking to them. The video then shows a group of people standing on a brick wall, with one person doing push-ups. The scene then shifts to a group of people sitting on a bench, with one person standing and talking to them. The video then s

The detected events are often highly descriptive, but they do not always capture the most important or salient moments in the video.

To address this, you may explore alternative strategies. One approach is to use a language model (LLM) to process frame-level descriptions generated from the video.

For example, you can experiment with randomly sampling frames from the video, generating descriptions for each frame using a vision-language model (VLM), and then aggregating these descriptions into a structured set of events using an LLM.


In [30]:
def sample_frames(video_path, num_frames):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        raise ValueError("Video has no frames.")

    frame_indices = sorted(random.sample(range(total_frames), min(num_frames, total_frames)))

    frames = []
    current_idx = 0
    target_idx_set = set(frame_indices)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if current_idx in target_idx_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame))

        current_idx += 1

    cap.release()
    return frames

In [32]:
# Build messages from sampled frames (standalone).
import os
import uuid
import cv2
from PIL import Image

video_path = "video_22.mp4"

if "processor" not in globals() or "model" not in globals():
    raise RuntimeError("processor or model not available; run the model-loading cell first")

def _quick_video_duration(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return (total / fps) if fps > 0 else 0.0

def _quick_sample_frames(path, interval_seconds=3):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames = []
    timestamps = []
    idx = 0
    next_t = 0.0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = idx / fps if fps > 0 else 0.0
        if t + 1e-3 >= next_t:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
            timestamps.append(t)
            next_t += interval_seconds
        idx += 1
    cap.release()
    return frames, timestamps

def _quick_save_frames(frames_with_ts, out_dir="outputs/tmp_frames"):
    os.makedirs(out_dir, exist_ok=True)
    saved = []
    for img, ts in frames_with_ts:
        name = f"frame_{uuid.uuid4().hex}.jpg"
        path = os.path.join(out_dir, name)
        img.save(path, format="JPEG", quality=90)
        saved.append((path, ts))
    return saved

duration = _quick_video_duration(video_path)
target_n = 6 if duration <= 120 else 10 if duration <= 300 else 14
interval = max(2, int(max(2, duration / (target_n * 1.5)))) if duration > 0 else 3

frames, timestamps = _quick_sample_frames(video_path, interval)
if len(frames) == 0:
    raise RuntimeError(f"No frames sampled from {video_path}; check the path and codecs")

saved = _quick_save_frames(list(zip(frames, timestamps)))
img_entries = [{"type": "image", "url": p} for p, _ in saved[:4]]

prompt_text = (
    "Describe the most relevant events in the video, list each event sequentially, "
    "using a numbered format. Describe it in terms of the actions different persons take"
)
messages = [{"role": "user", "content": img_entries + [{"type": "text", "text": prompt_text}]}]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=128)
generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
print(generated_texts[0])

User:















Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take
Assistant: In the first image, a man is performing sit-ups on a brick wall. He is wearing a black shirt and black shorts. Another person is climbing the wall, and a third person is standing nearby.

In the second image, a man is lying on the ground performing sit-ups. He is wearing a black shirt and black shorts. A person is climbing a brick wall in the background.

In the third image, a person is performing a handstand on a pedestal. The person is wearing a green shirt and black pants. The pedestal is located in front of a building with columns.

In


You may improve event detection by splitting the video into smaller temporal segments and prompting the VLM on each segment separately. The resulting partial event descriptions can then be combined into a single, coherent list using an LLM.

Crucially, you must define an evaluation scheme for event coverage, for example:
-How many annotated events are correctly detected?
-Which events are missed or incorrectly predicted?
-How does performance change with different prompting or segmentation strategies?

In [33]:
generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=256)

generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

KeyboardInterrupt: 

The generated outputs may be quite descriptive. While this can introduce noise, it also provides an opportunity to infer potential events through reasoning over these descriptions.

Therefore, one possible strategy is to first generate scene-level descriptions using a vision-language model (VLM), and then further process these descriptions using a language model (LLM) to extract structured events.

The choice of strategy is left to you. You are expected to design and implement an approach that produces meaningful event representations from the video.

In [ ]:
# Extended utilities: sampling, prompting, VLM runs, parsing, saving
import os
import json
import re
import random
from datetime import timedelta
from difflib import SequenceMatcher
import pandas as pd
import numpy as np

def _format_time(seconds):
    if seconds is None:
        return "00:00"
    td = timedelta(seconds=int(seconds))
    total_seconds = int(td.total_seconds())
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def compute_video_duration(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    if fps <= 0:
        return 0.0
    return frames / fps

def determine_target_events(duration_seconds):
    minutes = duration_seconds / 60.0
    if minutes <= 2:
        return random.randint(4,6)
    if 3 <= minutes <= 5:
        return random.randint(6,10)
    if 6 <= minutes <= 10:
        return random.randint(10,15)
    return random.randint(12,18)

def sample_frames_sparse(video_path, interval_seconds=3):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames = []
    timestamps = []
    if not cap.isOpened():
        raise IOError("Cannot open video: %s" % video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = total_frames / fps if fps>0 else 0
    next_t = 0.0
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        t = idx / fps if fps>0 else 0
        if t + 1e-3 >= next_t:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
            timestamps.append(t)
            next_t += interval_seconds
        idx += 1
    cap.release()
    return frames, timestamps

def _clean_numbered_text(raw_text):
    lines = [l.strip() for l in re.split(r'\n|\r', raw_text) if l.strip()]
    candidates = []
    for line in lines:
        pieces = re.split(r'\s*\d+\.\s*', line)
        if len(pieces) > 1:
            for p in pieces:
                p = p.strip()
                if p:
                    candidates.append(p)
        else:
            candidates.append(line)
    return candidates

def parse_events(raw_text):
    candidates = _clean_numbered_text(raw_text)
    events = []
    time_re = re.compile(r'(?P<start>\d{1,2}:\d{2}(?::\d{2})?)(?:\s*[–—~]\s*(?P<end>\d{1,2}:\d{2}(?::\d{2})?))?')
    for cand in candidates:
        m = time_re.search(cand)
        start = None
        end = None
        desc = cand
        if m:
            start = m.group('start')
            end = m.group('end')
            desc = (cand[:m.start()] + cand[m.end():]).strip(' ,.-:')
        desc = re.sub(r'^\(?P<num>\d+\)\s*', '', desc).strip()
        events.append({'description': desc, 'start_time': start, 'end_time': end})
    return events

def deduplicate_events(events, threshold=0.85):
    keep = []
    for e in events:
        desc = e['description']
        dup = False
        for k in keep:
            ratio = SequenceMatcher(None, desc.lower(), k['description'].lower()).ratio()
            if ratio >= threshold:
                dup = True
                break
        if not dup:
            keep.append(e)
    return keep

def save_outputs(video_path, raw_event_text, raw_ts_text, prompts, outputs_dir='outputs'):
    os.makedirs(outputs_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(video_path))[0]
    event_only_file = os.path.join(outputs_dir, f'{base}_event_only.txt')
    event_ts_file = os.path.join(outputs_dir, f'{base}_event_timestamps.txt')
    json_file = os.path.join(outputs_dir, f'{base}_vlm_results.json')
    with open(event_only_file, 'w', encoding='utf-8') as f:
        f.write(raw_event_text)
    with open(event_ts_file, 'w', encoding='utf-8') as f:
        f.write(raw_ts_text)
    payload = {'video': video_path, 'prompts': prompts, 'event_only': raw_event_text, 'event_timestamps': raw_ts_text}
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
    return event_only_file, event_ts_file, json_file

def _build_prompt(messages_frames, duration_sec, target_n, include_timestamps=False):
    duration_str = _format_time(duration_sec)
    header = (
        f'You are a helpful video understanding assistant. The video duration is {duration_str} (approx).',
        f'Please retrieve the most salient events from the video in sequential order.',
        f'Target number of events: {target_n}.',
        'Guidelines: avoid repetition, group similar short actions into a single event, focus on meaningful scene changes, and keep descriptions concise.',
    )
    if include_timestamps:
        header = header + ('Provide a start and end timestamp for each event in the format MM:SS or HH:MM:SS (e.g., 00:03-00:07).',)
    prompt_text = '\n'.join(header)
    content = [{'type':'text','text': prompt_text}]
    for i, (img, ts) in enumerate(messages_frames):
        tstr = _format_time(ts)
        content.append({'type':'text','text': f'Frame {i+1} timestamp: {tstr} (approx)'} )
        content.append({'type':'image','url': img})
    if include_timestamps:
        content.append({'type':'text','text':'Return results as a numbered list. Example: 1. Person enters room, 00:03-00:07'})
    else:
        content.append({'type':'text','text':'Return results as a numbered list of concise salient events only (no timestamps). Example: 1. Person enters room'})
    return content

def _run_model_on_messages(content, max_new_tokens=256):
    inputs = processor.apply_chat_template(content, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
    out = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return out

def run_vlm_event_only(video_path):
    duration = compute_video_duration(video_path)
    target_n = determine_target_events(duration)
    interval = max(2, int(max(2, duration / (target_n * 1.5)))) if duration>0 else 3
    frames, timestamps = sample_frames_sparse(video_path, interval)
    messages_frames = list(zip(frames, timestamps))
    content = _build_prompt(messages_frames, duration, target_n, include_timestamps=False)
    raw_out = _run_model_on_messages(content)
    parsed = parse_events(raw_out)
    dedup = deduplicate_events(parsed)
    df = pd.DataFrame([{'event_number': i+1, 'event_description': e['description']} for i,e in enumerate(dedup)])
    raw_ts_text = ''
    files = save_outputs(video_path, raw_out, raw_ts_text, {'prompt': content})
    return {'raw': raw_out, 'parsed': parsed, 'clean': dedup, 'df': df, 'files': files, 'duration': duration, 'target_n': target_n}

def run_vlm_event_with_timestamps(video_path):
    duration = compute_video_duration(video_path)
    target_n = determine_target_events(duration)
    interval = max(2, int(max(2, duration / (target_n * 1.5)))) if duration>0 else 3
    frames, timestamps = sample_frames_sparse(video_path, interval)
    messages_frames = list(zip(frames, timestamps))
    content = _build_prompt(messages_frames, duration, target_n, include_timestamps=True)
    raw_out = _run_model_on_messages(content, max_new_tokens=320)
    parsed = parse_events(raw_out)
    dedup = deduplicate_events(parsed)
    df = pd.DataFrame([{'event_number': i+1, 'event_description': e['description'], 'start_time': e['start_time'], 'end_time': e['end_time']} for i,e in enumerate(dedup)])
    files = save_outputs(video_path, '', raw_out, {'prompt': content})
    return {'raw': raw_out, 'parsed': parsed, 'clean': dedup, 'df': df, 'files': files, 'duration': duration, 'target_n': target_n}

# Usage example (call these cells manually after restarting kernel):
# res1 = run_vlm_event_only('video_1.mp4')
# res2 = run_vlm_event_with_timestamps('video_1.mp4')
# print('Raw event-only output:\n', res1['raw'])
# print('Cleaned events:\n', res1['df'])
# print('Raw event+timestamps output:\n', res2['raw'])
# print('Parsed timestamps:\n', res2['df'])


In [ ]:
# Run both pipelines on a sample video and save CSV outputs (run after restarting kernel and loading the model)
video_path = 'video_1.mp4'

if 'model' not in globals():
    raise RuntimeError('Model is not loaded. Run the model-loading cell (earlier in the notebook) and restart the kernel before executing this cell.')

try:
    print('Running event-only pipeline...')
    res1 = run_vlm_event_only(video_path)
    print(f"Duration: {res1['duration']:.2f}s, target events: {res1['target_n']}")
    print('Raw output (truncated):')
    print(res1['raw'][:1000])
    print('Cleaned events:')
    display(res1['df'])
    # Save cleaned dataframe as CSV next to the txt outputs
    ev_only_csv = res1['files'][0].replace('_event_only.txt','_event_only.csv')
    res1['df'].to_csv(ev_only_csv, index=False)
    print('Saved:', res1['files'], ev_only_csv)
except Exception as e:
    print('Error running event-only pipeline:', e)

try:
    print('
Running event-with-timestamps pipeline...')
    res2 = run_vlm_event_with_timestamps(video_path)
    print(f"Duration: {res2['duration']:.2f}s, target events: {res2['target_n']}")
    print('Raw output (truncated):')
    print(res2['raw'][:1000])
    print('Parsed timestamped events:')
    display(res2['df'])
    ev_ts_csv = res2['files'][1].replace('_event_timestamps.txt','_event_timestamps.csv')
    res2['df'].to_csv(ev_ts_csv, index=False)
    print('Saved:', res2['files'], ev_ts_csv)
except Exception as e:
    print('Error running event-with-timestamps pipeline:', e)

print('
Finished. Check the outputs/ folder for saved results.')